# Autonomous dual-arm wheelchair — current project

**Current workflow: sense a bottle on the table → plan an observed free-space approach → drive and brake → grasp → auxiliary support → remove and dispose of cap → primary regrasp → acknowledged, support-gated delivery.**

The autonomous workflow keeps one free-base MuJoCo scene and state. Both arms travel folded inside protective cases with closed covers, deploy only after braking, and return to their cases after successful delivery. The 6-DOF / 3-DOF arms, rear drive, actively steered front wheels and passive middle casters remain. This is a bounded flat-floor simulated task with ideal RGB-D/LiDAR, calibrated simulated localization, a short illustrative thread and a prescribed recipient hand. It does not establish arbitrary-room navigation or real passenger operation. See `docs/AUTONOMOUS_VALIDATION.md`. Earlier stationary and mobility experiments remain selectable below.

**Updated presentation:** simplified visual shells preserve collision geometry and mass. Live controller-driven window: `.venv/bin/python -B scripts/run_live.py --acknowledge-passenger`. Run robustness comparisons separately with `--robustness`; see `docs/IMPROVEMENTS.md`.


In [1]:
# Current focus: autonomous. Earlier comparisons remain selectable.
WORKFLOW = "autonomous"
RENDER = True
if WORKFLOW not in ("autonomous", "delivery", "grasp", "mobility", "all"):
    raise ValueError("Choose autonomous, delivery, grasp, mobility or all")
import os
import sys
from pathlib import Path
import subprocess
import importlib
import inspect

# Set the graphics backend before importing MuJoCo. Restart after changing it.
os.environ.setdefault("MUJOCO_GL", "cgl" if sys.platform == "darwin" else "egl")
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/model.py").is_file() and (p / "docs/PROJECT_CONTEXT.md").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the full project folder and run this notebook from it.")
# Keep this workspace first even if another project path is already present.
project_path = str(PROJECT_ROOT)
sys.path[:] = [project_path] + [p for p in sys.path if p != project_path]
OUTPUT = PROJECT_ROOT / "results/current"
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT / ".matplotlib"))
try:
    import mujoco
    import numpy as np
except ImportError as exc:
    raise RuntimeError("Select .venv/bin/python, or install requirements.txt in your kernel.") from exc
from IPython.display import Image, Markdown, display
import json
# Notebook kernels cache imports across edits. Refresh project modules in
# dependency order before binding functions used by later replay cells.
importlib.invalidate_caches()
import src
if Path(src.__file__).resolve().parent != PROJECT_ROOT / "src":
    raise RuntimeError("This kernel cached src from another project. Restart the kernel and Run All.")
for module_name in ("src.runtime", "src.design", "src.control", "src.arm_appearance", "src.model", "src.mobility_metrics", "src.simulation", "src.run_experiments", "src.manipulation", "src.task_scene", "src.grasp_scene", "src.run_grasp", "src.transfer", "src.stowage", "src.run_transfer", "src.assistive_task", "src.run_delivery", "src.navigation", "src.run_autonomous", "src.live_view", "src.robustness"):
    module = importlib.import_module(module_name)
    if Path(module.__file__).resolve().parent != PROJECT_ROOT / "src":
        raise RuntimeError(f"{module_name} came from another project. Restart the kernel and Run All.")
    importlib.reload(module)
from src import model as project_model
from src.model import MODEL, load_model, initialize
required_arguments = {"caster_outboard_offset", "clearance_geometry", "caster_resistance", "caster_tire_half_width"}
if not required_arguments.issubset(inspect.signature(project_model.load_model).parameters):
    raise RuntimeError("src/model.py is older than this notebook. Open the updated full project folder.")
from src.run_experiments import save_animation
print("Python:", sys.executable)
print("MuJoCo:", mujoco.__version__, "NumPy:", np.__version__)
print("Renderer:", os.environ["MUJOCO_GL"])
print("Output:", OUTPUT)
print("Model source:", project_model.__file__)
print("Model loader:", inspect.signature(project_model.load_model))

Python: /Users/ncssudaylolla/Downloads/autonomous_dual_arm_wheelchair_codex/.venv/bin/python
MuJoCo: 3.3.7 NumPy: 2.0.2
Renderer: cgl
Output: /Users/ncssudaylolla/Downloads/autonomous_dual_arm_wheelchair_codex/results/current
Model source: /Users/ncssudaylolla/Downloads/autonomous_dual_arm_wheelchair_codex/src/model.py
Model loader: (original_contacts=False, rocker_stiffness=None, caster_outboard_offset=0.0, clearance_geometry=False, caster_resistance='baseline', caster_tire_half_width=None, arm_appearance='baseline')


## Autonomous mobile task — continuous physics

The camera measures surface colors and ranges through pinhole rays; NumPy fits the blue bottle cylinder and verifies tabletop evidence. Environmental LiDAR rays update a conservative occupancy grid. Student-written A* and route simplification choose an approach; bounded Ackermann steering and independent rear wheel feedback execute it. Localization is an explicit calibrated simulated input, not SLAM. The route is forward-only and the experiment retains the tested clearance geometry and optional 40 mm middle tires.

Arrival requires perception, pose, speed, wheel-load, clearance and stability gates. Rear motor position braking holds the free chassis; there is **no base/world weld**. NumPy arm FK/Jacobians use the actual mobile root frame. The same bottle, cap, robot joints and physical state continue into manipulation. Release requires an explicit simulated acknowledgement and sustained measured recipient support. The palm first withdraws slightly from the supported bottle before the jaws open.

Comparisons include sideways passive casters, a small lateral approach offset, absent bottle, blocked route, absent acknowledgement and absent recipient support. Expected failures must inhibit the corresponding driving or passenger-release transition. These are bounded simulation demonstrations, not a certification of arbitrary caster starts, reverse driving, crowded-room navigation, real sensing or human safety.

**Arm storage update:** both arms have checked folded poses inside physical protective cases. Sliding covers must be confirmed closed and every arm collision envelope contained before driving. At the table, the stationary chair opens the covers and executes checked deployment. After recipient-supported delivery, the empty arms lift clear of the receiving hand, fold back into their cases, and close the covers. Missing acknowledgement/support retains the object and leaves its holding arm deployed. Covers add two accessory slide axes; the arms remain 6-DOF and 3-DOF. See `docs/STOWAGE_VALIDATION.md`.

In [ ]:
if WORKFLOW in ("autonomous", "all"):
    AUTONOMOUS_OUTPUT = PROJECT_ROOT / "results/autonomous_validation"
    # Explicit simulation test input, never inferred from the robot's progress.
    SIMULATED_PASSENGER_ACKNOWLEDGEMENT = True
    command = [sys.executable, "-B", "-m", "src.run_autonomous", "--compare", "--output", str(AUTONOMOUS_OUTPUT)]
    if SIMULATED_PASSENGER_ACKNOWLEDGEMENT:
        command.append("--acknowledge-passenger")
    if RENDER:
        command.append("--render")
    with subprocess.Popen(command, cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError("Autonomous comparison failed; inspect saved metrics and failed gates.")
    autonomous_report = json.loads((AUTONOMOUS_OUTPUT / "metrics.json").read_text())
    assert autonomous_report["nominal_passed"] and autonomous_report["all_expected_outcomes"]
    rows = ["| Trial | Navigation | Delivery / comparison outcome |", "|---|---|---|"]
    for label, trial in autonomous_report["trials"].items():
        outcome = "expected block verified" if trial.get("control_passed") else "passed"
        rows.append(f"| {label} | {trial['navigation']['passed']} | {outcome} |")
    display(Markdown(chr(10).join(rows)))
    nominal = autonomous_report["trials"]["nominal"]
    print("Navigation gates:", nominal["navigation"]["checks"])
    print("Parking stability:", nominal["parking"])
    print("Delivery gates:", nominal["delivery"]["checks"])
    display(Image(filename=str(AUTONOMOUS_OUTPUT / "nominal_navigation.png")))
    display(Image(filename=str(AUTONOMOUS_OUTPUT / "nominal_delivery.png")))
    display(Image(filename=str(AUTONOMOUS_OUTPUT / "nominal_storage.png")))
    if RENDER:
        display(Image(filename=str(AUTONOMOUS_OUTPUT / "arm_storage.png")))
        display(Image(filename=str(AUTONOMOUS_OUTPUT / "autonomous_stages.png")))
        display(Image(filename=str(AUTONOMOUS_OUTPUT / "nominal.gif")))


nominal navigation True None
nominal delivery True None
nominal idle stow confirmed
bottle_absent navigation False No supported bottle detected; driving and grasping inhibited
blocked_route navigation False No collision-free observed approach; driving and grasping inhibited
caster_sideways navigation True None
offset_approach navigation True None
no_acknowledgement navigation True None
no_acknowledgement delivery False No passenger acknowledgement: bottle retained by primary
  failed delivery gates: ['acknowledgement_before_release', 'recipient_support_before_release', 'completed_without_block', 'final_hold_observed', 'recipient_holds_bottle', 'grippers_unloaded', 'final_upright', 'final_position']
no_recipient_support navigation True None


In [ ]:
if WORKFLOW in ("autonomous", "all"):
    autonomous_tests = subprocess.run([sys.executable, "-B", "-m", "unittest", "tests.test_autonomous", "tests.test_stowage", "-v"],
                                    cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(autonomous_tests.stdout + autonomous_tests.stderr)
    autonomous_tests.check_returncode()


## Complete stationary bottle task and release interlocks

The bottle and cap remain independent free bodies. No gripper weld, runtime object pose reset or cap motor is used. A measured helical spring/damper model supplies equal-and-opposite thread forces until rotation and axial-rise conditions are met. The primary uses bounded NumPy pose IK, checked short Cartesian segments and repeated twist/regrasp cycles. The auxiliary stabilizes the bottle throughout opening.

Cap disposal requires measured support in the bin. Return transfer requires measured primary support before auxiliary release. The primary then corrects the measured bottle position, presents it at the passenger side, and waits for an explicit **simulated** acknowledgement. Only then does the hand surrogate approach. Primary release requires recipient support above 80% of bottle weight for 0.25 s. The last three seconds must show an upright bottle supported by the recipient, with both grippers unloaded and head/torso clearance maintained.

The comparison includes a jammed cap, no acknowledgement and absent recipient support. These must stop without passenger release. The cap model and prescribed hand are declared simulation assumptions; this is not validated navigation, human hand biomechanics, drinking or real passenger safety. Use `WORKFLOW="grasp"` to view the earlier 15-second retention studies.

In [ ]:
if WORKFLOW in ("delivery", "all"):
    DELIVERY_OUTPUT = PROJECT_ROOT / "results/delivery_validation"
    # Explicit test input. This does not represent a real passenger acknowledgement.
    SIMULATED_PASSENGER_ACKNOWLEDGEMENT = True
    command = [sys.executable, "-m", "src.run_delivery", "--compare", "--output", str(DELIVERY_OUTPUT)]
    if SIMULATED_PASSENGER_ACKNOWLEDGEMENT:
        command.append("--acknowledge-passenger")
    if RENDER:
        command.append("--render")
    with subprocess.Popen(command, cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError("Delivery comparison failed; inspect saved metrics.json and printed failed checks.")
    delivery_report = json.loads((DELIVERY_OUTPUT / "metrics.json").read_text())
    assert delivery_report["nominal_passed"] and delivery_report["negative_controls_blocked"]
    rows = ["| Trial | Physical delivery | Blocked reason |", "|---|---|---|"]
    for label, trial in delivery_report["trials"].items():
        rows.append(f"| {label} | {trial['passed']} | {trial['blocked_reason'] or 'none'} |")
    display(Markdown(chr(10).join(rows)))
    display(Image(filename=str(DELIVERY_OUTPUT / "delivery_comparison.png")))
    nominal = delivery_report["trials"]["nominal"]
    print("Measured stage events:")
    for event in nominal["events"]:
        print(event)
    print("All nominal checks:", nominal["checks"])
    if RENDER:
        display(Image(filename=str(DELIVERY_OUTPUT / "delivery_stages.png")))
        for label in ("nominal", "jammed_cap"):
            display(Markdown(f"**{label}: actual simulated robot, cap and recipient-hand poses**"))
            display(Image(filename=str(DELIVERY_OUTPUT / f"{label}.gif")))
        from PIL import Image as PillowImage
        with PillowImage.open(DELIVERY_OUTPUT / "nominal.gif") as animation:
            elapsed_ms = 0
            for frame in range(animation.n_frames):
                animation.seek(frame)
                elapsed_ms += animation.info.get("duration", 0)
        expected_ms = 100*len(np.load(DELIVERY_OUTPUT / "nominal_poses.npz")["qpos"])
        assert abs(elapsed_ms-expected_ms) <= 100
        print("Recorded GIF playback timing verified.")

In [ ]:
if WORKFLOW in ("delivery", "all"):
    # Integration tests independently audit CSV forces, release timing and pose recordings.
    # Source fingerprints force regeneration if evidence is stale after a code/model edit.
    delivery_tests = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_delivery.py", "-v"], cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(delivery_tests.stdout + delivery_tests.stderr)
    delivery_tests.check_returncode()

## Current focus: stationary bottle grasping

This section retains the earlier stationary experiment; the autonomous workflow above now integrates mobility. `WORKFLOW = "grasp"` in setup runs this section and skips the historical mobility experiments below. Select `"mobility"` or `"all"` when needed. All source remains in the same project and the canonical wheelchair XML is preserved.

The 6-DOF main arm and 3-DOF auxiliary arm remain attached. A declared world weld fixes the wheelchair base for this experiment; it does not represent validated wheelchair brakes. Two motor-driven finger slides replace the fixed primary fingers, and a free 0.3 kg rigid bottle rests on a table. The bottle has no weld, attachment, scripted pose or supporting constraint. The cap is only a visual marker at this stage.

NumPy implements FK, a geometric Jacobian, orientation-constrained damped-least-squares IK, sampled straight joint-path checking, quintic trajectories and saturated PD feedback. Engine kinematics/Jacobians only verify those algorithms. Payload compensation uses the NumPy COM Jacobian and the known simulated bottle mass; robot-only bias compensation uses MuJoCo as the previous verification controller did.

Sequence: settle → pregrasp → approach → close → confirm both pads → lift 80 mm → hold. Missing contact blocks lifting; excessive relative movement freezes the arm reference. The passenger proxy needs 10 mm clearance, arm references stay below 0.8 rad/s, jaw references below 0.04 m/s, primary torque limits remain 20/20/15/5/5/5 Nm, and jaw forces are capped at 8 N. Hold force is 6 N per jaw. No passenger delivery is performed.

The fixture adds two local wrist-housing assembly exclusions needed for bent wrist poses; external collisions remain enabled. Its explicit contact profile uses elliptic friction, `impratio=10`, three NoSlip iterations, pad/bottle `solref=(0.006,1)` and `solimp=(0.99,0.999,0.001)`. These are simulation assumptions, not calibrated hardware. The compliant-contact comparison retains its failures. See `docs/GRASP_VALIDATION.md`.


**Arm appearance updated from the user's photo:** compact rectangular pedestal, paired tapered upper-arm/forearm plates, distinct joint covers and compact wrist/hand. The six arm joints remain waist yaw, shoulder pitch, elbow pitch, wrist roll, wrist pitch and tool roll; the finger slides are separate tool DOFs. The active grasp fixture selects this reference-inspired style while the mobile baseline remains available.

Wheelchair-scale dimensions: upper arm **300 mm**, forearm **250 mm**, wrist-to-grasp **195 mm**, pedestal-to-shoulder **160 mm**, nominal shoulder height **881 mm**, side mounting offset **215 mm**. These retain the tested reach to the table; this is not an exact Rhino XR-3 reproduction or an ergonomic/hardware certification. Visual link plates fit inside the existing conservative collision cylinders, and joint transforms, masses, inertias and control limits are retained. See `docs/ARM_DESIGN.md` for dimensions, joint mapping and validation.


In [ ]:
if WORKFLOW in ("grasp", "all"):
    GRASP_OUTPUT = PROJECT_ROOT / "results/grasp_validation"
    grasp_run = subprocess.run([sys.executable, "-m", "src.run_grasp", "--compare", "--output", str(GRASP_OUTPUT)],
                               cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(grasp_run.stdout)
    if grasp_run.stderr:
        print(grasp_run.stderr)
    grasp_run.check_returncode()
    grasp_report = json.loads((GRASP_OUTPUT / "metrics.json").read_text())
    assert len(grasp_report["trials"]) == 6
    assert grasp_report["nominal_passed"]
    rows = ["| Trial | Minimum hold lift [mm] | Maximum relative drift [mm] | Tool error [mm] | All checks |",
            "| --- | ---: | ---: | ---: | --- |"]
    for label, trial in grasp_report["trials"].items():
        rows.append(f"| {label} | {1000*trial['minimum_lift_m']:.2f} | {1000*trial['max_hold_relative_slip_m']:.2f} | {1000*trial['max_hold_tool_error_m']:.2f} | {'PASS' if trial['passed'] else 'FAIL'} |")
    display(Markdown("\n".join(rows)))
    display(Image(filename=str(GRASP_OUTPUT / "grasp_comparison.png")))
    print("Known limitations: heavy payload, compliant contact, no payload compensation, open jaws and zero-friction cases remain recorded. Nominal success does not establish transfer or cap removal.")


In [ ]:
if WORKFLOW in ("grasp", "all") and RENDER:
    from src.grasp_scene import bottle_scene
    from src.run_grasp import render_grasp
    for label in ("nominal", "heavy", "zero_friction"):
        trial = grasp_report["trials"][label]
        replay_model = bottle_scene(mass=trial["mass_kg"], friction=trial["friction"],
                                   contact_impratio=trial["contact_impratio"], firm_contact=trial["firm_contact"], arm_appearance=trial.get("arm_appearance", "baseline"))
        poses = np.load(GRASP_OUTPUT / f"{label}_poses.npz")["qpos"]
        filename = GRASP_OUTPUT / f"{label}.gif"
        render_grasp(replay_model, poses, filename)
        display(Markdown(f"**Stationary grasp: {label}**"))
        display(Image(filename=str(filename)))

    preview = subprocess.run([sys.executable, "-m", "src.preview_arm"], cwd=PROJECT_ROOT,
                             capture_output=True, text=True)
    print(preview.stdout)
    if preview.stderr:
        print(preview.stderr)
    preview.check_returncode()
    display(Image(filename=str(PROJECT_ROOT / "results/arm_design/arm_comparison.png")))


In [ ]:
if WORKFLOW in ("grasp", "all"):
    grasp_tests = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_grasp.py", "-v"],
                                cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(grasp_tests.stdout)
    print(grasp_tests.stderr)
    grasp_tests.check_returncode()


## Fifteen-second holding, pickup clearance and table recovery

The same free bottle continues from pickup into transfer; only the wheelchair base is welded. The 6-DOF primary and 3-DOF auxiliary arm remain. The auxiliary starts in a declared waiting posture; NumPy pose IK, quintic trajectories and sampled collision checks select the routes.

The pregrasp waypoint is now **40 mm higher**. The old path grazed the nearer bottle before closing; the new open-jaw path requires at least 3 mm sampled clearance, and a separate runtime gate rejects preclose contact above 0.3 N. The retention comparison includes an explicit legacy-path replay at the nearer position.

Nominal, 15 mm nearer and 15 mm farther pickups continue through handover and a **15-second auxiliary-only hold**. A three-second trial uses the identical controller for comparison. Trajectory duration increases where required by the original 0.8 rad/s joint reference limit. Auxiliary contact and sustained upward support still gate handover release.

Open or zero-friction auxiliary jaws must block handover release. The primary retains support, opens and parks the auxiliary clear of the lowering path, and returns the bottle to the table. Primary release requires measured table support above 80% of bottle weight for 0.25 s. Final three-second table holding must show both grippers unloaded, an upright bottle, passenger clearance and original force/torque limits. Planning rejection retains primary support. Post-release loss still halts the simulation; autonomous recapture is not implemented.

The 0.6 kg challenge remains explicit. This is a stationary, known-pose experiment rather than cap removal, navigation or passenger operation. See `docs/TRANSFER_VALIDATION.md`.

In [ ]:
if WORKFLOW in ("grasp", "all"):
    TRANSFER_OUTPUT = PROJECT_ROOT / "results/transfer_validation"
    command = [sys.executable, "-m", "src.run_transfer", "--compare", "--retention", "--output", str(TRANSFER_OUTPUT)]
    if RENDER:
        command.append("--render")
    transfer_run = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(transfer_run.stdout)
    if transfer_run.stderr:
        print(transfer_run.stderr)
    transfer_run.check_returncode()
    transfer_report = json.loads((TRANSFER_OUTPUT / "metrics.json").read_text())
    assert transfer_report["positive_trials_passed"] and transfer_report["negative_controls_blocked"] and transfer_report["negative_controls_recovered"]
    rows = ["| Trial | Physical handover | Release blocked | Hold [s] | Drift [mm] | Table recovery |", "|---|---|---|---|---|---|"]
    for label, trial in transfer_report["trials"].items():
        drift = trial["max_hold_drift_m"]
        drift_text = "n/a" if drift is None else f"{1000*drift:.2f}"
        rows.append(f"| {label} | {trial['passed']} | {trial['release_blocked']} | {trial['hold_duration_s']:.2f} | {drift_text} | {trial['recovery']['passed']} |")
    display(Markdown(chr(10).join(rows)))
    display(Image(filename=str(TRANSFER_OUTPUT / "transfer_comparison.png")))
    display(Image(filename=str(TRANSFER_OUTPUT / "retention_comparison.png")))
    display(Image(filename=str(TRANSFER_OUTPUT / "recovery_comparison.png")))
    retention_report = json.loads((TRANSFER_OUTPUT / "retention_metrics.json").read_text())
    print("Retention results:", {name: trial["passed"] for name, trial in retention_report.items()})
    if RENDER:
        for label in ("nominal", "aux_zero_friction"):
            display(Markdown(f"**{label}: recorded continuous MuJoCo pickup and handover**"))
            display(Image(filename=str(TRANSFER_OUTPUT / f"{label}.gif")))
    if RENDER:
        from PIL import Image as PillowImage
        for label in ("nominal", "near", "aux_open", "aux_zero_friction"):
            with PillowImage.open(TRANSFER_OUTPUT / f"{label}.gif") as animation:
                elapsed_ms = 0
                for frame in range(animation.n_frames):
                    animation.seek(frame)
                    elapsed_ms += animation.info.get("duration", 0)
            expected_ms = 100*len(np.load(TRANSFER_OUTPUT / f"{label}_poses.npz")["qpos"])
            assert abs(elapsed_ms-expected_ms) <= 100, (label, elapsed_ms, expected_ms)
        print("GIF playback timing matches the recorded simulation samples.")
    if RENDER:
        for label in ("legacy_near", "near_7_5N"):
            display(Markdown(f"**Near-bottle pickup: {label}**"))
            display(Image(filename=str(TRANSFER_OUTPUT / f"retention_{label}.gif")))

In [ ]:
if WORKFLOW in ("grasp", "all"):
    transfer_tests = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_transfer.py", "-v"], cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(transfer_tests.stdout + transfer_tests.stderr)
    transfer_tests.check_returncode()

## 1. Configuration and safety scope

The 120 setting is an explicitly selected **mechanical spring comparison**, not a gain change. At 45 N m/rad the stowed chassis settles about 11° nose-down. The measured gravity moment is approximately 16.55 N m, giving an approximate requirement of `16.55 / (2 * radians(5)) = 94.8 N m/rad` per rocker; 120 provides margin and has been tested.

Provisional flat-floor checks: each arm error norm <1°, joint-limit excess <0.5°, chassis height 0.38–0.43 m, roll/pitch <5°, no unexpected deep contacts, and correct yaw direction. These are regression targets, not passenger-safety or steering-accuracy certification. Both arms stay stowed; there is no passenger handover motion. Set `RENDER = False` if offscreen graphics is unavailable.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    ROCKER_STIFFNESS = 120.0
    RENDER = True
    SCENARIOS = ("hold", "straight", "left", "right")
    model = load_model(rocker_stiffness=ROCKER_STIFFNESS)
    state, indices = initialize(model)
    print("nq / nv / actuators:", model.nq, model.nv, model.nu)
    print("Mass [kg]:", float(model.body_mass.sum()))
    print("Wheelbase / rear track / radius [m]:", indices["wheelbase"], indices["rear_track"], indices["rear_radius"])
    for i in range(model.nu):
        print(model.actuator(i).name, "->", model.joint(int(model.actuator_trnid[i, 0])).name)
    penetrations = [(model.geom(c.geom1).name, model.geom(c.geom2).name, float(c.dist))
                    for c in state.contact if c.dist < -0.001]
    print("Initial penetrations deeper than 1 mm:", penetrations)
    assert (model.nq, model.nv, model.nu) == (30, 29, 13)
    assert not penetrations


## 2. Controller and measurements

The active NumPy controller is `src/control.py`. Arm holding uses saturated PD plus MuJoCo model-bias feedforward: `tau = clip(Kp*(q_home-q) - Kd*qdot + bias, -limit, limit)`. The bias term is a physics verification baseline, not a student-derived dynamics algorithm.

Rear wheel targets use `curvature = tan(steer)/wheelbase` and `omega_left/right = (v/r)*(1 -/+ track*curvature/2)`. Wheelbase is derived from the model (0.82 m); the old 0.61 m yaw comparison is removed. Front wheels still receive shared steering targets. The current controller has unchanged gains.

Every simulation step refreshes post-step transforms and forces. Logs include arm errors, yaw, roll/pitch, steering, rear speeds and controls, joint-limit excess, six normal loads and contact-point longitudinal/lateral slip speeds. Slip is normal-force-weighted contact velocity in m/s, not a calibrated slip ratio.

Each six-second scenario compares **original contacts / 45 springs**, **corrected contacts / 45 springs**, and **corrected contacts / selected springs**. Original/contact-only posture failures remain visible in the report.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    # Run the canonical experiment entry point; numerical evidence is saved before rendering.
    run = subprocess.run([sys.executable, "-m", "src.run_experiments", "--compare",
                          "--rocker-stiffness", str(ROCKER_STIFFNESS), "--output", str(OUTPUT)],
                         cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(run.stdout)
    if run.stderr:
        print(run.stderr)
    run.check_returncode()
    report = json.loads((OUTPUT / "metrics.json").read_text())
    assert len(report["trials"]) == 12
    assert all(report["trials"]["corrected_" + s]["passed"] for s in SCENARIOS)


## 3. Quantitative results and graphs

Values below come from this execution. The original trials are expected to fail wrist-hold and posture criteria. Corrected contacts isolate the wrist improvement, and the optional spring comparison isolates the posture improvement. Passing these checks does not establish accurate curvature tracking.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    rows = ["| Trial | Main / auxiliary peak [deg] | Peak tilt [deg] | Turn yaw [deg] | Checks |",
            "| --- | ---: | ---: | ---: | --- |"]
    for name, metrics in report["trials"].items():
        a, b = metrics["peak_arm_error_norm_deg"]
        rows.append(f"| {name} | {a:.3f} / {b:.3f} | {metrics['peak_tilt_deg']:.3f} | {metrics['turn_yaw_deg']:.3f} | {'PASS' if metrics['passed'] else 'FAIL'} |")
    display(Markdown("\n".join(rows)))
    for scenario in SCENARIOS:
        display(Image(filename=str(OUTPUT / f"{scenario}_comparison.png")))


## 4. Animated MuJoCo comparisons

Animations replay the actual recorded joint positions, with a frame interval of 0.1 s. Only three representative GIFs are generated to avoid redundant files: original left turn, corrected left turn and corrected right turn. The numerical logs retain all 12 trials.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    if RENDER:
        for label in ("original_left", "corrected_left", "corrected_right"):
            metrics = report["trials"][label]
            replay_model = project_model.load_model(original_contacts=metrics["original_contacts"],
                                      rocker_stiffness=metrics["rocker_stiffness_Nm_rad"])
            poses = np.load(OUTPUT / f"{label}_poses.npz")["qpos"]
            filename = OUTPUT / f"{label}.gif"
            save_animation(replay_model, poses, filename)
            display(Markdown(f"**{label}**"))
            display(Image(filename=str(filename)))
    else:
        print("Rendering disabled; numerical logs, recorded poses and graphs remain available.")


## 5. Regression tests

The suite checks preserved DOFs and actuation, Ackermann turn-centre geometry, signed curvature and braking guards, wheel-axis geometry, narrow contact exclusions, external wrist collision detection, controller equivalence to the archived reference, synchronized timestamps, arm holding, chassis posture and ground-force balance. Tests execute MuJoCo physics.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    tests = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                           cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(tests.stdout)
    print(tests.stderr)
    tests.check_returncode()


## 6. Shared steering versus Ackermann, reverse and braking

This comparison keeps the same model, 120 N m/rad spring setting and PD gains. NumPy Ackermann targets use the front steering-axis positions measured relative to the rear-axle midpoint: `delta_i = atan2(x_i*kappa, 1-y_i*kappa)`. Inner and outer angles differ; the original shared-angle controller remains the default.

Nine scenarios are run with each steering mode: hold, forward straight/left/right, reverse straight/left/right, forward braking and reverse braking. The dedicated braking command changes from ±0.18 m/s to zero at 3 s.

Curvature is measured as integrated chassis yaw rate divided by integrated **signed achieved longitudinal velocity at the rear axle**, in the fixed 2.5–3.4 s window. The window must cover at least 3 cm of signed travel. The provisional gates are <20% curvature error and <20% steady speed error. Braking requires prior motion >0.05 m/s, settling within 1 s and 5 cm, and low residual body/wheel speed. A 0.2 s quiet interval prevents a transient zero crossing being counted as a stop.

**Posture checks and full mobility checks are reported separately.** A reproducible experiment can complete while the accuracy criteria fail. This notebook intentionally displays those failures. No gains, friction or collision geometry are retuned to make Ackermann appear successful.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    STEERING_OUTPUT = PROJECT_ROOT / "results/steering_validation"
    comparison = subprocess.run([sys.executable, "-m", "src.run_experiments", "--steering-compare",
                                 "--rocker-stiffness", str(ROCKER_STIFFNESS), "--output", str(STEERING_OUTPUT)],
                                cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(comparison.stdout)
    if comparison.stderr:
        print(comparison.stderr)
    comparison.check_returncode()
    steering_report = json.loads((STEERING_OUTPUT / "metrics.json").read_text())
    assert len(steering_report["trials"]) == 18
    assert steering_report["regression_passed"]
    print("Full mobility validation:", "PASS" if steering_report["mobility_passed"] else "INCOMPLETE — inspect failures below")
    lines = ["| Trial | Speed ratio | Curvature error | Stop time / distance | Full mobility |",
             "| --- | ---: | ---: | ---: | --- |"]
    for label, trial in steering_report["trials"].items():
        motion = trial["mobility"]
        ratio = motion.get("steady_speed_ratio")
        curvature = motion["curvature"]
        braking = motion["braking"]
        ratio_text = "—" if ratio is None else f"{ratio:.3f}"
        curvature_text = "—" if curvature is None else f"{100*curvature['relative_error']:.1f}%"
        stopping_text = "—"
        if braking is not None and braking["stopping_time_s"] is not None:
            stopping_text = f"{braking['stopping_time_s']:.3f} s / {100*braking['stopping_distance_m']:.2f} cm"
            if not motion["checks"].get("moving_before_brake", False):
                stopping_text += " (already slow)"
        lines.append(f"| {label} | {ratio_text} | {curvature_text} | {stopping_text} | {'PASS' if trial['mobility_passed'] else 'FAIL'} |")
    display(Markdown(chr(10).join(lines)))
    for label, trial in steering_report["trials"].items():
        failed = [name for name, passed in trial["mobility"]["checks"].items() if not passed]
        if failed:
            print(label, failed)


## 7. Tracking plots and new animations

Dotted lines in the tracking figures are targets. Pointwise curvature is masked below 0.03 m/s; reported curvature uses the guarded integral above. Low reverse-turn speed can give apparently good curvature while failing motion requirements. Braking measurements start at the actual control transition; settling is observed at 0.02 s log resolution.

Three additional animations show Ackermann forward turning, reverse turning and reverse braking. The previous section's corrected-left animation is the shared-angle baseline. Per-joint arm errors/torques, caster angles and all six wheel spin speeds are also logged in the CSV files.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    for scenario in ("left", "right", "reverse_left", "brake", "reverse_brake"):
        display(Image(filename=str(STEERING_OUTPUT / f"{scenario}_tracking.png")))
    if RENDER:
        for label in ("ackermann_left", "ackermann_reverse_left", "ackermann_reverse_brake"):
            trial = steering_report["trials"][label]
            replay_model = project_model.load_model(rocker_stiffness=trial["rocker_stiffness_Nm_rad"])
            poses = np.load(STEERING_OUTPUT / f"{label}_poses.npz")["qpos"]
            filename = STEERING_OUTPUT / f"{label}.gif"
            save_animation(replay_model, poses, filename)
            display(Markdown(f"**{label}**"))
            display(Image(filename=str(filename)))


## 8. Bounded integral feedback: isolate steering, then rear drive

The original P/D gains and motor limits remain unchanged. This comparison adds torque-valued integral terms with conditional anti-windup: integrate only when error is not pushing an already saturated command further into saturation, and clamp stored torque separately. Steering Ki = 80 N m/(rad s), bound = 8 N m; rear-speed Ki = 7 N m/rad, bound = 10 N m. Both use a nominal Kp/Ki = 1 s accumulation timescale, selected from the measured steady offsets rather than increasing motor limits.

Fresh state is created per trial. Rear I memory resets at stop/direction changes; steering I resets on discrete reference changes greater than 0.05 rad. The two staged variants add steering integration first, then steering plus drive integration. Arms retain their existing controller.

A new contact check records loaded non-ground contacts even when penetration is less than 1 mm. Shallow wheel/frame blocking can therefore fail full mobility acceptance without failing the historical deep-contact check. Shared steering and PD remain the defaults.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    INTEGRAL_OUTPUT = PROJECT_ROOT / "results/integral_validation"
    integral_run = subprocess.run([sys.executable, "-m", "src.run_experiments", "--integral-compare",
        "--steering-mode", "ackermann", "--rocker-stiffness", str(ROCKER_STIFFNESS), "--output", str(INTEGRAL_OUTPUT)],
        cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(integral_run.stdout)
    if integral_run.stderr:
        print(integral_run.stderr)
    integral_run.check_returncode()
    integral_report = json.loads((INTEGRAL_OUTPUT / "metrics.json").read_text())
    assert len(integral_report["trials"]) == 27
    lines = ["| Trial | Speed ratio | Curvature error | Full mobility |", "| --- | ---: | ---: | --- |"]
    for label, trial in integral_report["trials"].items():
        motion = trial["mobility"]
        speed = motion.get("steady_speed_ratio")
        curvature = motion["curvature"]
        speed_text = "—" if speed is None else f"{speed:.3f}"
        error_text = "—" if curvature is None else f"{100*curvature['relative_error']:.2f}%"
        lines.append(f"| {label} | {speed_text} | {error_text} | {'PASS' if trial['mobility_passed'] else 'FAIL'} |")
    display(Markdown(chr(10).join(lines)))
    for scenario in ("left", "right", "reverse_left"):
        display(Image(filename=str(INTEGRAL_OUTPUT / f"{scenario}_tracking.png")))
    display(Image(filename=str(INTEGRAL_OUTPUT / "left_integral.png")))
    if RENDER:
        label = "steering_drive_pi_left"
        trial = integral_report["trials"][label]
        replay_model = project_model.load_model(rocker_stiffness=trial["rocker_stiffness_Nm_rad"])
        poses = np.load(INTEGRAL_OUTPUT / f"{label}_poses.npz")["qpos"]
        filename = INTEGRAL_OUTPUT / f"{label}.gif"
        save_animation(replay_model, poses, filename)
        display(Image(filename=str(filename)))


## 9. Longer reverse tests and caster clearance diagnosis

Twelve-second tests halve the command speed and extend the steering phase from 1.5 to 8.5 s. The late measurement windows are fixed at 7.5–8.4 s for curvature and 7.5–8.0 s for speed; early metrics remain in the JSON. Initial passive-caster orientations are 0°, 180°, 90°, and opposed ±90°. These are diagnostic initial conditions, not a hidden caster actuator or an implemented alignment maneuver.

The 90° state exposes a middle-wheel/bogie overlap in the original geometry. A full nominal swivel sweep also exposes this obstruction. Loaded contacts can block turning even at shallow penetration.

A separately labelled **exploratory geometry comparison** moves each middle-caster mount outward by 0.09 m in memory, preserving wheel radius, trail, friction and all collision masks. The swept horizontal envelope estimate is `sqrt((0.045+0.12)^2 + 0.05^2) = 0.1724 m`; nominal bogie outer edge is 0.305 m. The original pivot at 0.40 m has insufficient clearance. The offset tests a pivot at 0.49 m. This widens the middle track and requires a corresponding bracket/CAD design; it is not a validated mechanical replacement. The original XML remains unchanged. Dynamic tests can still fail through other contacts and alignment behavior.

The diagnostic CLI may exit 1 because the original geometry fails. Complete saved trial records and failure reports are required below; exceptions or incomplete suites remain errors.

In [ ]:
if WORKFLOW in ("mobility", "all"):
    caster_reports = {}
    for label, offset in (("original", 0.0), ("clearance_experiment", .09)):
        output = PROJECT_ROOT / "results" / ("caster_validation" if offset == 0 else "caster_clearance_validation")
        result = subprocess.run([sys.executable, "-m", "src.run_experiments", "--caster-compare",
            "--steering-mode", "ackermann", "--feedback-mode", "steering_drive_pi",
            "--rocker-stiffness", str(ROCKER_STIFFNESS), "--caster-outboard-offset", str(offset), "--output", str(output)],
            cwd=PROJECT_ROOT, capture_output=True, text=True)
        print(label, result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode not in (0, 1):
            result.check_returncode()
        recorded = json.loads((output / "metrics.json").read_text())
        assert len(recorded["trials"]) == 8 and "regression_passed" in recorded
        assert result.returncode == (0 if recorded["regression_passed"] else 1)
        caster_reports[label] = recorded
        lines = ["| Trial | Late speed ratio | Late curvature error | Loaded contacts | Full mobility |",
                 "| --- | ---: | ---: | ---: | --- |"]
        for name, trial in recorded["trials"].items():
            motion = trial["mobility"]
            curvature = motion["curvature"]
            error_text = "—" if curvature is None else f"{100*curvature['relative_error']:.1f}%"
            loaded = sum(c["peak_normal_force_N"] >= 1. for c in trial["non_ground_contacts"])
            lines.append(f"| {name} | {motion.get('steady_speed_ratio', 0):.3f} | {error_text} | {loaded} | {'PASS' if trial['mobility_passed'] else 'FAIL'} |")
        display(Markdown(chr(10).join(lines)))
        display(Image(filename=str(output / "reverse_left_integral.png")))
    if RENDER:
        output = PROJECT_ROOT / "results/caster_validation"
        for name in ("caster_0_reverse_left", "caster_180_reverse_left"):
            trial = caster_reports["original"]["trials"][name]
            replay_model = project_model.load_model(rocker_stiffness=trial["rocker_stiffness_Nm_rad"],
                                      caster_outboard_offset=trial["caster_outboard_offset_m"])
            poses = np.load(output / f"{name}_poses.npz")["qpos"]
            filename = output / f"{name}.gif"
            save_animation(replay_model, poses, filename)
            display(Markdown(f"**Diagnostic: {name} — inspect reported contacts before interpreting motion**"))
            display(Image(filename=str(filename)))


## 10. Supported geometry candidate: passive casters retained

The candidate moves the middle swivel axes outward by 90 mm per side and front steering axes by 50 mm. Four collision-enabled overhead supports add 1 kg total. Collision-enabled forks provide 7 mm nominal tire clearance. Wheel sizes, 45 mm caster trail, passive unlimited caster swivel/rolling joints, damping, friction, controller gains and motor limits are retained. Ackermann targets use the measured new front-axis positions. The canonical XML and default geometry remain unchanged; this optional design is not adopted as a finished wheelchair.

The audit samples 32,850 configurations per variant: each side's rocker/bogie ranges at five positions, front steering at nine positions, and caster swivel every 5 degrees. Each side is checked separately with the other neutral, arms stowed and ground excluded. A 5 mm minimum is required. This is a sampled clearance check, not a proof for every continuous configuration, simultaneous cross-side articulation, obstacles or passenger loads.

The width tradeoff is significant: middle pivot track grows from 0.80 to 0.98 m; neutral tire width becomes 1.08 m and the full swivel envelope is about 1.325 m. Fork/bearing mass properties are coarse simulation assumptions requiring mechanical/CAD review.


In [ ]:
if WORKFLOW in ("mobility", "all"):
    GEOMETRY_OUTPUT = PROJECT_ROOT / "results/geometry_validation"
    sweep_run = subprocess.run([sys.executable, "-m", "src.geometry_audit", "--output", str(GEOMETRY_OUTPUT)],
                               cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(sweep_run.stdout)
    if sweep_run.stderr:
        print(sweep_run.stderr)
    sweep_run.check_returncode()
    clearance_report = json.loads((GEOMETRY_OUTPUT / "clearance.json").read_text())
    assert clearance_report["variants"]["supported_candidate"]["passed"]
    display(Image(filename=str(GEOMETRY_OUTPUT / "clearance.png")))
    geometry_reports = {}
    for family, flags, count in (("standard", ["--geometry-compare"], 18),
                                 ("caster", ["--caster-compare", "--clearance-geometry"], 8)):
        output = GEOMETRY_OUTPUT / family
        result = subprocess.run([sys.executable, "-m", "src.run_experiments", *flags,
            "--steering-mode", "ackermann", "--feedback-mode", "steering_drive_pi",
            "--rocker-stiffness", str(ROCKER_STIFFNESS), "--require-mobility-pass", "--output", str(output)],
            cwd=PROJECT_ROOT, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        # These experiments deliberately expose unresolved passive-alignment failures.
        # Require complete reports and their failure exit code, not a silent successful run.
        assert result.returncode == 1, (result.returncode, result.stderr)
        geometry_reports[family] = json.loads((output / "metrics.json").read_text())
        assert len(geometry_reports[family]["trials"]) == count
    rows = ["| Trial | Speed / command | Curvature error | Loaded self-contact | Full gates |",
            "| --- | ---: | ---: | --- | --- |"]
    for family, report_g in geometry_reports.items():
        for label, trial in report_g["trials"].items():
            motion = trial["mobility"]
            curvature = motion.get("curvature")
            error = f"{100*curvature['relative_error']:.2f}%" if curvature else "insufficient motion"
            ratio = motion.get("steady_speed_ratio")
            speed = f"{ratio:.3f}" if ratio is not None else "n/a"
            loaded = not motion["checks"]["no_loaded_self_contacts"]
            rows.append(f"| {family}/{label} | {speed} | {error} | {loaded} | {'PASS' if trial['mobility_passed'] else 'FAIL'} |")
    display(Markdown("\n".join(rows)))
    for family, filename in (("standard", "reverse_left_tracking.png"),
                             ("caster", "reverse_left_integral.png")):
        display(Image(filename=str(GEOMETRY_OUTPUT / family / filename)))


In [ ]:
if WORKFLOW in ("mobility", "all"):
    if RENDER:
        selections = [(PROJECT_ROOT / "results/caster_validation", "caster_180_reverse_left", False),
                      (GEOMETRY_OUTPUT / "caster", "caster_180_reverse_left", True),
                      (GEOMETRY_OUTPUT / "caster", "caster_90_reverse_left", True)]
        for output, label, candidate in selections:
            trial = json.loads((output / "metrics.json").read_text())["trials"][label]
            replay_model = project_model.load_model(rocker_stiffness=trial["rocker_stiffness_Nm_rad"], clearance_geometry=candidate)
            poses = np.load(output / f"{label}_poses.npz")["qpos"]
            filename = output / f"{label}.gif"
            save_animation(replay_model, poses, filename)
            display(Markdown(f"**{'Supported candidate' if candidate else 'Baseline'}: {label}**"))
            display(Image(filename=str(filename)))
    display(Markdown("The supported candidate clears the sampled wheel/fork envelope and removes loaded frame contacts in these trials. Pre-aligned reverse turns pass; other caster starting orientations still fail motion gates. The caster orientations above are initial conditions, not motorized alignment. General reverse mobility remains unfinished."))


## 11. Passive alignment: resistance, contact scrub and a steering maneuver

This stage keeps the middle wheels unactuated and unlimited. A seven-variant comparison runs both reverse directions from 0°, 180°, 90° and opposed starts (56 trials). Existing controller gains/limits, geometry supports, 120 rocker setting, 12 s protocol, half speed and late acceptance windows remain fixed.

Variants: baseline; bearing damping/friction reduced fivefold; middle rolling damping reduced fivefold; both reductions; a 20° reverse steering pulse during the first 1.5 s; a narrower middle tire (40 mm full width instead of 100 mm); and that narrower tire with the same pulse. The physical changes are **sensitivity hypotheses, not measured hardware parameters**. Narrow tires retain radius, trail and mass, with matching visual/collision widths and analytically updated cylinder inertia. Current profile defaults are retained.

The new logs separate caster trail torque from the opposing tire-contact moment, and record caster rates, passive/constraint torques, wheel loads, rear traction utilization and contact slip. `condim=3` contacts have no explicit torsional/rolling friction channels, but forces at multiple contact points can still create a resisting moment. See [MuJoCo contact documentation](https://mujoco.readthedocs.io/en/latest/modeling.html#contact-parameters).

The pulse uses only existing front-steering/rear-drive motors at existing speed/torque limits, with no pose resets or hidden caster actuator. It changes the first straight segment into a curve, so it needs obstacle clearance; it is an open-loop bounded experiment, not a validated general alignment supervisor. Standard forward/reverse/braking and sampled articulated clearance are checked separately on the narrow candidate. Full independent swivel width remains about 1.312 m.


In [ ]:
if WORKFLOW in ("mobility", "all"):
    ALIGNMENT_OUTPUT = PROJECT_ROOT / "results/alignment_validation"
    checks = [("matrix", ["-m", "src.run_experiments", "--profile", "current", "--alignment-compare"], ALIGNMENT_OUTPUT, 56),
              ("narrow_standard", ["-m", "src.run_experiments", "--profile", "current", "--scenario", "mobility",
                                   "--caster-tire-half-width", "0.02"], ALIGNMENT_OUTPUT / "narrow_standard", 9)]
    alignment_reports = {}
    for family, arguments, output, count in checks:
        run = subprocess.run([sys.executable, *arguments, "--output", str(output)],
                             cwd=PROJECT_ROOT, capture_output=True, text=True)
        print(run.stdout)
        if run.stderr:
            print(run.stderr)
        recorded = json.loads((output / "metrics.json").read_text())
        assert len(recorded["trials"]) == count
        assert run.returncode == (0 if recorded["mobility_passed"] else 1)
        alignment_reports[family] = recorded
    rows = ["| Variant | Full mobility passes / trials |", "| --- | ---: |"]
    for variant in ("baseline", "low_bearing", "low_roll", "low_both", "steer_pulse", "narrow_tire", "narrow_pulse"):
        trials = [trial for label, trial in alignment_reports["matrix"]["trials"].items() if label.startswith(variant+"_caster_")]
        rows.append(f"| {variant} | {sum(t['mobility_passed'] for t in trials)} / {len(trials)} |")
    display(Markdown("\n".join(rows)))
    rows = ["| Narrow tire + pulse | Speed / command | Curvature error | Full gates |",
            "| --- | ---: | ---: | --- |"]
    for label, trial in alignment_reports["matrix"]["trials"].items():
        if label.startswith("narrow_pulse_"):
            curvature = trial["mobility"]["curvature"]
            error = f"{100*curvature['relative_error']:.2f}%" if curvature else "insufficient motion"
            rows.append(f"| {label} | {trial['mobility'].get('steady_speed_ratio', 0):.3f} | {error} | {'PASS' if trial['mobility_passed'] else 'FAIL'} |")
    display(Markdown("\n".join(rows)))
    for scenario in ("reverse_left", "reverse_right"):
        display(Image(filename=str(ALIGNMENT_OUTPUT / f"{scenario}_diagnostics.png")))
    geometry_run = subprocess.run([sys.executable, "-m", "src.geometry_audit", "--caster-tire-half-width", "0.02",
                                  "--output", str(ALIGNMENT_OUTPUT / "geometry")],
                                 cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(geometry_run.stdout)
    if geometry_run.stderr:
        print(geometry_run.stderr)
    geometry_run.check_returncode()
    display(Image(filename=str(ALIGNMENT_OUTPUT / "geometry/clearance.png")))


In [ ]:
if WORKFLOW in ("mobility", "all"):
    if RENDER:
        for label in ("baseline_caster_90_reverse_left", "narrow_tire_caster_90_reverse_left", "narrow_pulse_caster_0_reverse_left"):
            trial = alignment_reports["matrix"]["trials"][label]
            replay_model = project_model.load_model(rocker_stiffness=trial["rocker_stiffness_Nm_rad"],
                clearance_geometry=trial["clearance_geometry"], caster_resistance=trial["caster_resistance"],
                caster_tire_half_width=trial["caster_tire_half_width_m"] if trial["caster_tire_half_width_m"] != .05 else None)
            poses = np.load(ALIGNMENT_OUTPUT / f"{label}_poses.npz")["qpos"]
            filename = ALIGNMENT_OUTPUT / f"{label}.gif"
            save_animation(replay_model, poses, filename)
            display(Markdown(f"**{label}**"))
            display(Image(filename=str(filename)))
    display(Markdown("These results apply to the stated low-speed, flat-floor, stowed-arm tests. Sensitivity parameters and the pulse remain optional; wider operating conditions, arbitrary caster orientations, passenger loading, obstacle clearance and narrow indoor passage remain unvalidated. Standard reverse turns without the pulse must still be assessed separately."))


## 12. Progress, limitations and retained history

The wrist correction reduced driving main-arm peak error from about 9.52° to 0.28° without gain changes. With the optional 120 springs, the tested peak chassis tilt is about 3.2° and both arms remain below 0.32°. This execution recalculates these values rather than hard-coding acceptance.

**Still outstanding:** validated speed/curvature tracking in both directions, reverse-turn slowdown, connecting collision-free planning to tracking, navigation and sensing, bottle grasp/transfer/cap physics, passenger clearance and acknowledgement.

The earlier student NumPy FK, geometric Jacobian, DLS IK, RRT, cubic/quintic trajectories and fixed-base PD experiments are retained in `references/history.zip`, especially `notebooks/coursework_updated.ipynb`. Their reported physics/control limitations remain unresolved; they are not silently merged into the mobile controller. The archive also preserves former models, reference samples and the original audit results.

Read `docs/CONTACT_VALIDATION.md` for equations and tested limitations, `docs/PROGRESS_AUDIT.md` for the initial assessment, and `docs/PROJECT_CONTEXT.md` for the full goal. Generated files for this notebook are in `results/current/`.

Bounded PI now passes the tested forward-turn speed/curvature gates. Reverse motion remains limited by swivel clearance and loaded self-contacts. Neither the PI mode nor the exploratory caster mount shift is silently adopted as the default.

Latest geometry validation is in `docs/GEOMETRY_VALIDATION.md`. The supported candidate remains optional because passive alignment and footprint constraints are unresolved.